Vector Store → simple similarity search system

Vector Database → full retrieval system with indexing, filtering, and scaling

1️⃣ Vector Store

A vector store is a component or library that stores embeddings and performs similarity search.
It is usually simple and embedded inside an application.

Characteristics
    Stores vector embeddings
    Performs similarity search
    Often runs locally
    Limited database features

Examples
    FAISS
    Chroma
    Annoy


2️⃣ Vector Database

A vector database is a full database system designed to store and query vectors at scale.

It includes database capabilities like indexing, scaling, filtering, APIs, and distributed storage.

Characteristics
1-Stores billions of vectors
2.Distributed architecture
3.Metadata filtering
4.High scalability
5.Production-ready

Examples

Pinecone ,
Milvus,
Weaviate,
Qdrant

diffrent vector database implementations

In [61]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [62]:

loader = TextLoader(r"doc/data.txt")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)

chunks = splitter.split_documents(docs)
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

 # faiss

In [63]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [64]:
vectorstore = FAISS.from_documents(chunks , embeddings)
vectorstore.add_documents(chunks)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 2})

In [65]:
results = retriever.invoke("types of plant")
for i in results:
    print(i.page_content)
    print("-"*20)

3. TYPES OF PLANTS
- Trees: Large plants with a single woody stem or trunk.
- Shrubs: Woody plants smaller than trees, typically with multiple stems.
- Herbs: Non-woody plants with soft stems.
- Grasses: Monocotyledonous plants with narrow leaves.
- Ferns: Plants that reproduce via spores, not seeds.
- Succulents/Cacti: Plants with thick, fleshy parts for water storage.
- Aquatic Plants: Plants adapted to living in water.

4. IMPORTANCE OF PLANTS
- Oxygen Production: Plants are the primary source of oxygen for living organisms.
- Food Source: They provide fruits, vegetables, grains, and nuts.
- Ecosystem Services: They prevent soil erosion, maintain atmospheric balance, and create habitats.
- Resources: Plants provide wood, medicine, fiber, and raw materials.

5. BASIC CARE
- Sunlight: Required for energy.
- Water: Essential for hydration and nutrient transport.
- Soil/Nutrients: Provide structural support and minerals.
--------------------
Research: Healthline notes that while many pl

# Chroma

ChromaDB is an embedding-based vector database used to store and search text using semantic similarity.

It is commonly used in:

RAG pipelines

LLM applications

Document search

Chatbots

In [66]:
from langchain_community.vectorstores import Chroma

# Notes : 
if you use  vectorstore.add_documents(chunks), you usually do not need Chroma.from_documents(). Both methods insert documents into Chroma 

from_documents() → create DB + insert documents -creates the vector store and adds documents at the same time.

add_documents() → insert into existing DB


Running locally (In-Memory)

a Chroma server running in memory by simply instantiating a Chroma instance with a collection name and your embeddings

In [67]:
vectorstore = Chroma.from_documents(
    collection_name="sample_collections",
    embedding=embeddings,
   documents=chunks,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})
results = retriever.invoke("types of plant")
for i in results:
    print(i.page_content)
    print("-"*20)

3. TYPES OF PLANTS
- Trees: Large plants with a single woody stem or trunk.
- Shrubs: Woody plants smaller than trees, typically with multiple stems.
- Herbs: Non-woody plants with soft stems.
- Grasses: Monocotyledonous plants with narrow leaves.
- Ferns: Plants that reproduce via spores, not seeds.
- Succulents/Cacti: Plants with thick, fleshy parts for water storage.
- Aquatic Plants: Plants adapted to living in water.

4. IMPORTANCE OF PLANTS
- Oxygen Production: Plants are the primary source of oxygen for living organisms.
- Food Source: They provide fruits, vegetables, grains, and nuts.
- Ecosystem Services: They prevent soil erosion, maintain atmospheric balance, and create habitats.
- Resources: Plants provide wood, medicine, fiber, and raw materials.

5. BASIC CARE
- Sunlight: Required for energy.
- Water: Essential for hydration and nutrient transport.
- Soil/Nutrients: Provide structural support and minerals.
--------------------


In [68]:
print("Document count:", vectorstore._collection.count())

Document count: 5


Running locally (with data persistence)

You can provide the persist_directory argument to save your data across multiple runs of your program:

In ChromaDB, persist_directory is used to store the vector database on disk so that it does not disappear after the program stops.

If you don’t use persist_directory, the vector store is in-memory only and will be deleted when the program ends.

In [69]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    collection_name="sample_collections",
    persist_directory="./sample_chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

results = retriever.invoke("types of plant")

for i in results:
    print(i.page_content)
    print("-"*20)

3. TYPES OF PLANTS
- Trees: Large plants with a single woody stem or trunk.
- Shrubs: Woody plants smaller than trees, typically with multiple stems.
- Herbs: Non-woody plants with soft stems.
- Grasses: Monocotyledonous plants with narrow leaves.
- Ferns: Plants that reproduce via spores, not seeds.
- Succulents/Cacti: Plants with thick, fleshy parts for water storage.
- Aquatic Plants: Plants adapted to living in water.

4. IMPORTANCE OF PLANTS
- Oxygen Production: Plants are the primary source of oxygen for living organisms.
- Food Source: They provide fruits, vegetables, grains, and nuts.
- Ecosystem Services: They prevent soil erosion, maintain atmospheric balance, and create habitats.
- Resources: Plants provide wood, medicine, fiber, and raw materials.

5. BASIC CARE
- Sunlight: Required for energy.
- Water: Essential for hydration and nutrient transport.
- Soil/Nutrients: Provide structural support and minerals.
--------------------


In [70]:
print(vectorstore._collection.count())

5


4️⃣ Loading Existing Persistent DB

In [72]:

vectorstore = Chroma(
    collection_name="sample_collections",
    persist_directory="./sample_chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

results = retriever.invoke("types of plant")

for i in results:
    print(i.page_content)

3. TYPES OF PLANTS
- Trees: Large plants with a single woody stem or trunk.
- Shrubs: Woody plants smaller than trees, typically with multiple stems.
- Herbs: Non-woody plants with soft stems.
- Grasses: Monocotyledonous plants with narrow leaves.
- Ferns: Plants that reproduce via spores, not seeds.
- Succulents/Cacti: Plants with thick, fleshy parts for water storage.
- Aquatic Plants: Plants adapted to living in water.

4. IMPORTANCE OF PLANTS
- Oxygen Production: Plants are the primary source of oxygen for living organisms.
- Food Source: They provide fruits, vegetables, grains, and nuts.
- Ecosystem Services: They prevent soil erosion, maintain atmospheric balance, and create habitats.
- Resources: Plants provide wood, medicine, fiber, and raw materials.

5. BASIC CARE
- Sunlight: Required for energy.
- Water: Essential for hydration and nutrient transport.
- Soil/Nutrients: Provide structural support and minerals.


In [73]:
print(vectorstore._collection.count())

5


In [ ]:
#2️⃣ vectorstore.add_documents() (Add Later)

vectorstore = Chroma(
    collection_name="plant_collection",
    persist_directory="./chroma_db"
)

vectorstore.add_documents(chunks)


Connecting to a chroma Server


Add items to vector store
Update items in vector store